In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from train import ExpressionPerformer

sns.set_theme(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [26]:
RUN_DIR = ROOT / "checkpoints_performer/20jo1hdd"
CHECKPOINT_PATH = RUN_DIR / "best_model.pt"
TCGA_MATRIX_PATH = ROOT / "data/tcga/tcga_unstranded_matrix.parquet"
TCGA_TPM_MATRIX_PATH = ROOT / "data/tcga/tcga_tpm_unstranded_matrix.parquet"
TCGA_METADATA_PATH = ROOT / "data/tcga/tcga_metadata.parquet"
TCGA_FILE_MAPPING_PATH = ROOT / "data/tcga/file_mapping.csv"
GENE_ORDER_PATH = ROOT / "data/archs4/train_orthologs/canonical_genes.csv"

assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
assert TCGA_MATRIX_PATH.exists(), f"Missing TCGA matrix: {TCGA_MATRIX_PATH}"
assert TCGA_FILE_MAPPING_PATH.exists(), f"Missing file mapping: {TCGA_FILE_MAPPING_PATH}"
assert GENE_ORDER_PATH.exists(), f"Missing gene order file: {GENE_ORDER_PATH}"


def build_tcga_value_matrix(file_mapping, gene_order, value_column, cache_path=None):
    file_mapping = file_mapping.copy()
    sample_tables = []

    for row in file_mapping.itertuples(index=False):
        file_path = ROOT / row.file_path
        if not file_path.exists():
            raise FileNotFoundError(f"Missing TCGA raw file: {file_path}")

        sample_df = pd.read_csv(file_path, sep="\t", skiprows=1)
        sample_df = sample_df.loc[sample_df["gene_name"].notna(), ["gene_name", value_column]].copy()
        sample_df = sample_df.drop_duplicates("gene_name", keep="first")
        sample_series = sample_df.set_index("gene_name")[value_column]
        sample_series = sample_series.reindex(gene_order, fill_value=0.0).astype(np.float32)
        sample_series.name = row.file_id
        sample_tables.append(sample_series)

    matrix = pd.DataFrame(sample_tables)
    matrix.index.name = "sample_id"
    matrix.columns = gene_order
    matrix = matrix.astype(np.float32)

    if cache_path is not None:
        matrix.to_parquet(cache_path)

    return matrix


def load_or_build_tcga_tpm_matrix(gene_order):
    if TCGA_TPM_MATRIX_PATH.exists():
        tpm_df = pd.read_parquet(TCGA_TPM_MATRIX_PATH)
        tpm_df = tpm_df.reindex(columns=gene_order, fill_value=0.0)
        return tpm_df.astype(np.float32)

    file_mapping = pd.read_csv(TCGA_FILE_MAPPING_PATH)
    print(f"Building TPM cache from {len(file_mapping)} raw TCGA files...")
    return build_tcga_value_matrix(
        file_mapping=file_mapping,
        gene_order=gene_order,
        value_column="tpm_unstranded",
        cache_path=TCGA_TPM_MATRIX_PATH,
    )


def load_tcga_inputs():
    raw_df = pd.read_parquet(TCGA_MATRIX_PATH)
    gene_order_df = pd.read_csv(GENE_ORDER_PATH)
    gene_order = gene_order_df["gene_symbol"].tolist()
    raw_df = raw_df.reindex(columns=gene_order, fill_value=0.0).astype(np.float32)
    tpm_df = load_or_build_tcga_tpm_matrix(gene_order)

    if TCGA_METADATA_PATH.exists():
        meta = pd.read_parquet(TCGA_METADATA_PATH)
    else:
        meta = pd.read_csv(TCGA_FILE_MAPPING_PATH)

    rename_map = {}
    if "file_id" in meta.columns:
        rename_map["file_id"] = "sample_id"
    if "project_id" not in meta.columns and "Project ID" in meta.columns:
        rename_map["Project ID"] = "project_id"
    if "tissue_type" not in meta.columns and "Tissue Type" in meta.columns:
        rename_map["Tissue Type"] = "tissue_type"
    meta = meta.rename(columns=rename_map)

    required = {"sample_id", "project_id", "tissue_type"}
    missing = required.difference(meta.columns)
    if missing:
        raise ValueError(f"Metadata is missing columns: {sorted(missing)}")

    meta = meta.set_index("sample_id")
    raw_df.index = raw_df.index.astype(str)
    tpm_df.index = tpm_df.index.astype(str)
    meta.index = meta.index.astype(str)

    shared_index = meta.index.intersection(raw_df.index).intersection(tpm_df.index)
    meta = meta.loc[shared_index].copy()
    raw_df = raw_df.loc[shared_index].copy()
    tpm_df = tpm_df.loc[shared_index].copy()
    return raw_df, tpm_df, meta, gene_order


def build_expression_performer(cfg, num_genes, device):
    model = ExpressionPerformer(
        num_genes=num_genes,
        hidden_dim=cfg["hidden_dim"],
        n_heads=cfg["num_heads"],
        n_layers=cfg["num_layers"],
        ffn_dim=cfg["ffn_dim"],
        ree_base=cfg["ree_base"],
        mask_token_id=cfg.get("mask_token", -10),
        feature_type=cfg.get("feature_type", "sqr"),
        compute_type=cfg.get("compute_type", "iter"),
    )
    model = model.to(device)
    model.eval()
    return model


def encode_hidden_states(model, x):
    _, num_genes = x.shape
    gene_ids = torch.arange(num_genes, device=x.device)
    gene_emb = model.gene_embedding(gene_ids)
    ree_emb = model.ree(x)
    hidden = gene_emb.unsqueeze(0) + ree_emb
    for layer in model.layers:
        rfs = layer.attention.sample_rfs(x.device)
        hidden = layer.full_forward(hidden, rfs)
    return hidden


def predict_expression(model, expr_array, batch_size=16, device=device):
    expr_tensor = torch.tensor(expr_array, dtype=torch.float32)
    loader = DataLoader(TensorDataset(expr_tensor), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            preds.append(model(x_batch).detach().cpu())
    return torch.cat(preds, dim=0).numpy()


def extract_transcriptome_embeddings(expr_array, model, aggregate_type="all", batch_size=16, device=device):
    expr_tensor = torch.tensor(expr_array, dtype=torch.float32)
    loader = DataLoader(TensorDataset(expr_tensor), batch_size=batch_size, shuffle=False)
    emb_list = []
    model.eval()
    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            hidden = encode_hidden_states(model, x_batch).detach().cpu().numpy()
            if aggregate_type == "max":
                pooled = hidden.max(axis=1)
            elif aggregate_type == "mean":
                pooled = hidden.mean(axis=1)
            elif aggregate_type == "median":
                pooled = np.median(hidden, axis=1)
            elif aggregate_type == "all":
                pooled = hidden.max(axis=1) + hidden.mean(axis=1) + np.median(hidden, axis=1)
            else:
                raise ValueError(f"Unsupported aggregate_type: {aggregate_type}")
            emb_list.append(pooled)
    return torch.tensor(np.vstack(emb_list), dtype=torch.float32)

In [27]:
payload = torch.load(CHECKPOINT_PATH, map_location="cpu")
cfg = payload["config"]
X_counts, X_tpm, meta, gene_order = load_tcga_inputs()

expected_num_genes = int(payload["model_state_dict"]["gene_embedding.weight"].shape[0])
assert len(gene_order) == expected_num_genes, (
    f"Gene dictionary mismatch: mapping has {len(gene_order)} genes but checkpoint expects {expected_num_genes}"
)

X_counts = X_counts.sort_index()
X_tpm = X_tpm.loc[X_counts.index].copy()
meta = meta.loc[X_counts.index].copy()
X_log = np.log1p(np.maximum(X_tpm.values.astype(np.float32), 0.0))

model = build_expression_performer(cfg, num_genes=X_log.shape[1], device=device)
model.load_state_dict(payload["model_state_dict"])
model.eval()

summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "run_dir": str(RUN_DIR),
    "val_loss": float(payload.get("val_loss", np.nan)),
    "epoch": int(payload.get("epoch", -1)),
    "total_params": int(payload.get("total_params", -1)),
    "num_samples": int(X_counts.shape[0]),
    "num_genes": int(X_counts.shape[1]),
    "hidden_dim": int(cfg["hidden_dim"]),
    "num_heads": int(cfg["num_heads"]),
    "num_layers": int(cfg["num_layers"]),
    "ffn_dim": int(cfg["ffn_dim"]),
    "normalization": cfg.get("normalization"),
    "gene_dictionary": str(GENE_ORDER_PATH),
    "tcga_input_transform": "log1p(tpm_unstranded)",
    "tcga_tpm_source": str(TCGA_TPM_MATRIX_PATH),
}

summary

{'checkpoint': '/home/walt/Attention/checkpoints_performer/20jo1hdd/best_model.pt',
 'run_dir': '/home/walt/Attention/checkpoints_performer/20jo1hdd',
 'val_loss': 0.6107481718063354,
 'epoch': 5,
 'total_params': 19323905,
 'num_samples': 3481,
 'num_genes': 15165,
 'hidden_dim': 512,
 'num_heads': 8,
 'num_layers': 4,
 'ffn_dim': 2048,
 'normalization': 'log1p_tpm',
 'gene_dictionary': '/home/walt/Attention/data/archs4/train_orthologs/canonical_genes.csv',
 'tcga_input_transform': 'log1p(tpm_unstranded)',
 'tcga_tpm_source': '/home/walt/Attention/data/tcga/tcga_tpm_unstranded_matrix.parquet'}

In [28]:
A = "TCGA-BRCA"
B = "TCGA-LUAD"

samples_A = meta.index[meta["project_id"] == A].tolist()
samples_B = meta.index[meta["project_id"] == B].tolist()

n = min(len(samples_A), len(samples_B))
samples_A = samples_A[:n]
samples_B = samples_B[:n]
subset_samples = samples_A + samples_B

X_sub_counts = X_counts.loc[subset_samples].values.astype(np.float32)
X_sub_tpm = X_tpm.loc[subset_samples].values.astype(np.float32)
X_sub = np.log1p(np.maximum(X_sub_tpm, 0.0))
labels = np.array([A] * n + [B] * n)

print(f"{A}: {len(samples_A)} samples")
print(f"{B}: {len(samples_B)} samples")
print(f"Balanced subset: {len(subset_samples)} samples ({n} per cohort)")
print(f"Subset matrix shape: {X_sub.shape}")
print("Model input transform: log1p(tpm_unstranded)")

In [29]:
def evaluate_imputation(model, expr_array, mask_token, masking_ratio=0.15, batch_size=16, seed=42):
    rng = np.random.default_rng(seed)
    expr = np.asarray(expr_array, dtype=np.float32).copy()
    mask = rng.random(expr.shape) < masking_ratio
    masked_expr = expr.copy()
    masked_expr[mask] = mask_token

    preds = predict_expression(model, masked_expr, batch_size=batch_size, device=device)
    true_vals = expr[mask]
    pred_vals = preds[mask]

    return {
        "masked_fraction": float(mask.mean()),
        "pearson": float(pearsonr(true_vals, pred_vals)[0]),
        "spearman": float(spearmanr(true_vals, pred_vals)[0]),
    }

imputation_metrics = evaluate_imputation(
    model,
    X_sub,
    mask_token=cfg.get("mask_token", -10),
    masking_ratio=cfg.get("mask_ratio", 0.15),
    batch_size=16,
    seed=42,
)

imputation_metrics

{'masked_fraction': 0.1499042973218062,
 'pearson': 0.8379296987289225,
 'spearman': 0.8341749271848985}

In [30]:
sample_embs_subset = extract_transcriptome_embeddings(
    X_sub,
    model=model,
    aggregate_type="all",
    batch_size=16,
    device=device,
).numpy()

X_raw = X_sub_counts
X_pca_raw = PCA(n_components=2).fit_transform(X_raw)
pca_log = PCA(n_components=2)
X_pca_log = pca_log.fit_transform(X_sub)
pca_emb = PCA(n_components=2)
X_pca_emb = pca_emb.fit_transform(sample_embs_subset)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x=X_pca_raw[:, 0], y=X_pca_raw[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[0])
axes[0].set_title(f"PCA Raw Counts: {A} vs {B}")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

sns.scatterplot(x=X_pca_log[:, 0], y=X_pca_log[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[1])
axes[1].set_title(f"PCA log1p(tpm_unstranded): {A} vs {B}")
axes[1].set_xlabel(f"PC1 ({pca_log.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca_log.explained_variance_ratio_[1]:.1%})")

sns.scatterplot(x=X_pca_emb[:, 0], y=X_pca_emb[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[2])
axes[2].set_title(f"PCA ExpressionPerformer Embeddings: {A} vs {B}")
axes[2].set_xlabel(f"PC1 ({pca_emb.explained_variance_ratio_[0]:.1%})")
axes[2].set_ylabel(f"PC2 ({pca_emb.explained_variance_ratio_[1]:.1%})")

plt.tight_layout()
plt.show()

In [33]:
tsne_raw = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="random", random_state=42)
X_tsne_raw = tsne_raw.fit_transform(X_raw)

tsne_log = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="random", random_state=42)
X_tsne_log = tsne_log.fit_transform(X_sub)

tsne_emb = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="random", random_state=42)
X_tsne_emb = tsne_emb.fit_transform(sample_embs_subset)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x=X_tsne_raw[:, 0], y=X_tsne_raw[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[0])
axes[0].set_title(f"t-SNE Raw Counts: {A} vs {B}")
axes[0].set_xlabel("tSNE1")
axes[0].set_ylabel("tSNE2")

sns.scatterplot(x=X_tsne_log[:, 0], y=X_tsne_log[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[1])
axes[1].set_title(f"t-SNE log1p(tpm_unstranded): {A} vs {B}")
axes[1].set_xlabel("tSNE1")
axes[1].set_ylabel("tSNE2")

sns.scatterplot(x=X_tsne_emb[:, 0], y=X_tsne_emb[:, 1], hue=labels, alpha=0.75, s=45, ax=axes[2])
axes[2].set_title(f"t-SNE ExpressionPerformer Embeddings: {A} vs {B}")
axes[2].set_xlabel("tSNE1")
axes[2].set_ylabel("tSNE2")

plt.tight_layout()
plt.show()

In [32]:
def eval_model(features, y, name, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    clf = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    )
    scores = cross_val_score(clf, features, y, cv=cv, scoring="f1_macro")
    return {
        "features": name,
        "f1_macro_mean": float(scores.mean()),
        "f1_macro_std": float(scores.std()),
    }

pca_dim = min(64, X_sub.shape[0] - 1, X_sub.shape[1])
X_pca64 = PCA(n_components=pca_dim, random_state=42).fit_transform(X_sub)

results = pd.DataFrame(
    [
        eval_model(sample_embs_subset, labels, "ExpressionPerformer Embeddings"),
        eval_model(X_sub, labels, "log1p(tpm_unstranded)"),
        eval_model(X_pca64, labels, f"PCA({pca_dim}) on log1p(tpm_unstranded)"),
    ]
).sort_values("f1_macro_mean", ascending=False)

results

,features,f1_macro_mean,f1_macro_std
1,log1p(tpm_unstranded),0.998337,0.002037
2,PCA(64) on log1p(tpm_unstranded),0.986696,0.008873
0,ExpressionPerformer Embeddings,0.940141,0.026269
